## Sensibilidad a las condiciones iniciales — versión combinada

Las dos celdas originales calculaban **la misma trayectoria logística** dos veces
(una vez para superponer las dos trayectorias, otra vez para graficar su distancia),
cada una con su propio par de sliders `r` / `ε`.

Aquí se calcula la trayectoria **una sola vez** y se generan las dos vistas
(trayectorias superpuestas y distancia) a partir del mismo resultado, controladas
por un único par de sliders y apiladas verticalmente con `alt.vconcat`.

Se agregó además un botón (`ToggleButton`) que aplica el logaritmo **natural**
(`ln`) a los datos del panel de distancia (no solo un cambio de escala del eje).
Esto es intencional: la fórmula de referencia es

$$\ln|x'_t - x_t| \approx \lambda t + c$$

así que, con el toggle activo, la **pendiente** de la curva resultante es una
estimación directa del exponente de Lyapunov `λ`. Con `r` en zona estable
(por ejemplo `r=2.51`) la pendiente sale negativa (las trayectorias convergen);
con `r` en zona caótica (por ejemplo `r=3.9`) sale positiva (las trayectorias
se separan).

Cuando las dos trayectorias colapsan al mismo valor exacto en `float64`
(algo que ocurre en régimen convergente, por límite de precisión numérica),
`ln(0)` no existe: en vez de inventar un piso arbitrario o cortar la curva
ahí, esos puntos puntuales se **saltean** y la curva sigue dibujándose con
los puntos válidos que vengan después (el título del panel lo indica cuando
esto ocurre). Esos huecos son un límite de precisión de la máquina, no un
cambio real en la dinámica.

In [1]:
import numpy as np
import pandas as pd
import altair as alt
import ipywidgets as widgets

from IPython.display import display, clear_output


# -----------------------------------
# Logistic trajectory
# -----------------------------------

def logistic_trajectory(x0, r, n):
    values = [x0]
    for _ in range(n):
        values.append(r * values[-1] * (1 - values[-1]))
    return values


# -----------------------------------
# Shared controls
# -----------------------------------

r_slider = widgets.FloatSlider(
    value=3.7,
    min=2.5,
    max=3.99,
    step=0.01,
    description="r:",
    continuous_update=False,
    readout_format=".2f"
)

epsilon_slider = widgets.FloatLogSlider(
    value=0.001,
    base=10,
    min=-5,      # 10^-5
    max=-1,      # 10^-1
    step=0.1,
    description="\u03b5:",
    continuous_update=False
)

log_scale_toggle = widgets.ToggleButton(
    value=False,
    description="Log scale (distance)",
    tooltip="Toggle log scale on the distance panel"
)

plot_output = widgets.Output()


# -----------------------------------
# Draw both charts (stacked) from one shared computation
# -----------------------------------

def draw_charts(*args):

    r = r_slider.value
    epsilon = epsilon_slider.value

    x0_1 = 0.10
    x0_2 = x0_1 + epsilon
    n_iter = 50

    traj_1 = np.array(logistic_trajectory(x0_1, r, n_iter))
    traj_2 = np.array(logistic_trajectory(x0_2, r, n_iter))
    difference = np.abs(traj_1 - traj_2)

    t = np.arange(n_iter + 1)

    # --- Data for the trajectories panel ---
    df_traj = pd.DataFrame({
        "t": np.concatenate([t, t]),
        "x": np.concatenate([traj_1, traj_2]),
        "Initial condition": (
            [f"x\u2080 = {x0_1:.6f}"] * (n_iter + 1)
            + [f"x\u2080 = {x0_2:.6f}"] * (n_iter + 1)
        )
    })

    # --- Data for the difference panel ---
    use_log = log_scale_toggle.value

    if use_log:
        # ln(0) doesn't exist. Instead of flooring with an arbitrary small
        # number (fake plateau) or cutting the curve at the first zero
        # (which would hide any later non-zero values, e.g. in a chaotic
        # regime where distance can dip to float64 zero and later grow
        # again), we simply drop the exact-zero points and keep plotting
        # everything else -- gaps in the line mark precision limits, not
        # a change in the dynamics.
        valid = difference != 0.0
        t_used = t[valid]
        diff_used = difference[valid]
        plotted_difference = np.log(diff_used)

        df_diff = pd.DataFrame({
            "t": t_used,
            "difference": plotted_difference,
            "raw_difference": diff_used
        })
        n_skipped = int((~valid).sum())
    else:
        df_diff = pd.DataFrame({
            "t": t,
            "difference": difference,
            "raw_difference": difference
        })
        n_skipped = 0

    # --- Top panel: the two trajectories ---
    chart_traj = (
        alt.Chart(df_traj)
        .mark_line(point=True, strokeWidth=2)
        .encode(
            x=alt.X("t:Q", title="Iteration"),
            y=alt.Y("x:Q", title="x\u209c", scale=alt.Scale(domain=[0, 1])),
            color=alt.Color("Initial condition:N", title="Initial condition"),
            strokeDash=alt.StrokeDash("Initial condition:N", legend=None),
            tooltip=[
                "Initial condition:N",
                alt.Tooltip("t:Q", title="Iteration"),
                alt.Tooltip("x:Q", format=".8f")
            ]
        )
        .properties(
            width=750,
            height=280,
            title=f"Trajectories \u2014 r = {r:.2f}, \u03b5 = {epsilon:.6f}"
        )
    )

    # --- Bottom panel: their absolute distance ---
    diff_title = (
        "ln(|x\u209c\u00b9 \u2212 x\u209c\u00b2|)"
        if use_log else
        "|x\u209c\u00b9 \u2212 x\u209c\u00b2|"
    )

    diff_chart_title = "Distance between trajectories"
    if n_skipped > 0:
        diff_chart_title += f" ({n_skipped} point(s) at exact 0 skipped: precision limit, not dynamics)"

    chart_diff = (
        alt.Chart(df_diff)
        .mark_line(point=True, strokeWidth=2, color="firebrick")
        .encode(
            x=alt.X("t:Q", title="Iteration"),
            y=alt.Y(
                "difference:Q",
                title=diff_title
                # always linear now: when use_log is True the *data* is
                # already ln-transformed, so the axis stays linear and the
                # slope you read off the line is a direct estimate of lambda
            ),
            tooltip=[
                alt.Tooltip("t:Q", title="Iteration"),
                alt.Tooltip("raw_difference:Q", title="Distance", format=".8f"),
                alt.Tooltip("difference:Q", title="ln(Distance)" if use_log else "Distance", format=".6f")
            ]
        )
        .properties(
            width=750,
            height=280,
            title=diff_chart_title
        )
    )

    combined = alt.vconcat(chart_traj, chart_diff).resolve_scale(color="independent")

    with plot_output:
        clear_output(wait=True)
        display(combined)


# -----------------------------------
# Connect sliders (shared by both plots)
# -----------------------------------

r_slider.observe(draw_charts, names="value")
epsilon_slider.observe(draw_charts, names="value")
log_scale_toggle.observe(draw_charts, names="value")

display(
    widgets.HBox([r_slider, epsilon_slider, log_scale_toggle]),
    plot_output
)

draw_charts()


Output()